In [ ]:
import pandas as pd
import numpy as np
import ast
import os
from collections import defaultdict
from scipy.stats import pearsonr, spearmanr
from tqdm import tqdm

In [ ]:
mean_df = pd.read_csv("/jdata/qmy/VirtualCell/diff_matrix_predict/drug_cell_diff_5.0uM.csv", index_col=[0,1])
pseudo_df = pd.read_csv("/jdata/qmy/VirtualCell/diff_matrix_predict/sc_pseudo_bulk_10_diff_5.0uM.csv", index_col=[0,1])
pred_df = pd.read_csv("/jdata/qmy/VirtualCell/diff_matrix_predict/sc_predictions_5.0uM.csv", index_col=[0,1])

In [ ]:
common_samples = mean_df.index.intersection(pseudo_df.index).intersection(pred_df.index)
genes = mean_df.columns.tolist()
gene_to_idx = {g: i for i, g in enumerate(genes)}
kegg_genesets = {}
with open('/jdata/qmy/c2.all.v2026.1.Hs.symbols.gmt.txt', 'r') as f:
    for i, line in enumerate(f):
        if i >= 200:  
            break
        parts = line.strip().split('\t')
        if len(parts) >= 3:
            kegg_genesets[parts[0]] = set(parts[2:])

In [ ]:
sample_expr = {}

for sample in common_samples:
    mean_vals = mean_df.loc[sample].values
    pseudo_vals = pseudo_df.loc[sample].values
    pred_vals = pred_df.loc[sample].values
    
    sample_expr[sample] = {
        'pred': pred_vals,
        'sc': pseudo_vals,
        'mean': mean_vals
    }
genes = mean_df.columns.tolist()
gene_to_idx = {g: i for i, g in enumerate(genes)}

In [ ]:
results = []
for geneset_name, gene_set in tqdm(kegg_genesets.items(), desc="Processing pathways"):
    gene_set_filtered = [g for g in gene_set if g in gene_to_idx]
    if len(gene_set_filtered) < 10:
        continue
    
    gene_indices = [gene_to_idx[g] for g in gene_set_filtered]
    
    pearson_pred_sc = []
    pearson_sc_mean = []
    
    for sample in common_samples:
        pred_vals = sample_expr[sample]['pred'][gene_indices]
        sc_vals = sample_expr[sample]['sc'][gene_indices]
        mean_vals = sample_expr[sample]['mean'][gene_indices]
        
        # 检查变异
        if np.std(pred_vals) > 1e-10 and np.std(sc_vals) > 1e-10:
            r, _ = pearsonr(pred_vals, sc_vals)
            pearson_pred_sc.append(r)
        
        if np.std(sc_vals) > 1e-10 and np.std(mean_vals) > 1e-10:
            r, _ = pearsonr(sc_vals, mean_vals)
            pearson_sc_mean.append(r)
    
    if len(pearson_pred_sc) > 0 and len(pearson_sc_mean) > 0:
        results.append({
            'pathway': geneset_name,
            'n_genes': len(gene_set_filtered),
            'pearson_pred_sc': np.mean(pearson_pred_sc),
            'pearson_sc_mean': np.mean(pearson_sc_mean)
        })
result_df = pd.DataFrame(results)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
top_n = 200
top_pathways = result_df.nlargest(top_n, 'pearson_sc_mean')[['pathway', 'pearson_pred_sc', 'pearson_sc_mean']]
fig, ax = plt.subplots(figsize=(14, 8))
x = range(len(top_pathways))
width = 0.35
ax.bar([i - width/2 for i in x], top_pathways['pearson_pred_sc'], width, 
       label='pred vs sc', alpha=0.7, color='blue')
ax.bar([i + width/2 for i in x], top_pathways['pearson_sc_mean'], width, 
       label='sc vs mean', alpha=0.7, color='red')
ax.set_xticks(x)
ax.set_xticklabels(top_pathways['pathway'], rotation=90, fontsize=8)
ax.set_ylabel('Pearson Correlation')
ax.set_title(f'Top {top_n} Pathways: pred-sc vs sc-mean Correlation')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()